# 03 · Applications of LLMs
### *Aligning & Deploying LLMs — Unit 2*

One aligned model, many jobs. We drive a small **instruction-tuned** model (`google/flan-t5-base`) with different
prompts to cover the five headline applications from the deck — plus purpose-built models where they shine:

**Text generation · Question answering · Translation · Summarization · Coding**

> CPU is fine for these small models; a GPU just makes them snappier.

In [ ]:
!pip -q install "transformers>=4.40" datasets sentencepiece

In [ ]:
from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

flan = pipeline("text2text-generation", model="google/flan-t5-base", device=device)

def ask(prompt, **kw):
    return flan(prompt, max_new_tokens=kw.get("max_new_tokens", 80))[0]["generated_text"]

## 1 · Text generation

In [ ]:
print(ask("Write a two-sentence product blurb for a reusable steel water bottle."))
print("---")
print(ask("Brainstorm three creative names for a coffee shop near a library."))

## 2 · Question answering

Two flavours: **generative** QA (the model writes the answer) and **extractive** QA (the model points to a span
in a given context — the pattern behind retrieval-augmented generation / RAG).

In [ ]:
# Generative (closed-book)
print("Generative:", ask("Answer the question. Question: Why is the sky blue? Answer:"))

# Extractive (open-book) — the basis of RAG
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=device)
context = ("The Eiffel Tower was completed in 1889 for the World's Fair in Paris. "
           "It stands 330 metres tall and was the tallest structure in the world until 1930.")
print("Extractive:", qa(question="How tall is the Eiffel Tower?", context=context)["answer"])

## 3 · Translation

FLAN can translate via a prompt; dedicated **OPUS-MT** models are stronger for a specific pair.

In [ ]:
print("FLAN:", ask("Translate to French: The weather is beautiful today."))

trans = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr", device=device)
print("OPUS-MT:", trans("The weather is beautiful today.")[0]["translation_text"])

## 4 · Summarization

In [ ]:
article = ('''
Large language models have moved from research labs into everyday products. After pretraining on web-scale text,
they are refined with instruction tuning and human feedback, then compressed so they can run affordably. Companies
now choose between hosted APIs, cloud GPUs, on-premise servers, and lightweight local runtimes depending on their
needs for privacy, cost, and control.
''')
summ = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", device=device)
print(summ(article, max_length=60, min_length=20)[0]["summary_text"])

## 5 · Coding

General instruction models can write simple code; **code-specialized** models do it far better. Here is a small one.

In [ ]:
code_gen = pipeline("text-generation", model="Salesforce/codegen-350M-mono", device=device)
prompt = "# Python function that returns the nth Fibonacci number\ndef fib(n):\n"
print(code_gen(prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"])

## Recap & your turn

- The **same model** covers many applications — the *prompt* selects the task.
- **Extractive QA** over a supplied context is the seed of **RAG**: retrieve relevant text, then let the LLM answer from it.
- **Specialized** models (OPUS-MT, CodeGen) beat a generalist on their niche.

**Exercises**
1. Build a tiny **RAG** loop: given 3 short documents, pick the most relevant with embeddings, then feed it to `qa`.
2. Compare `flan-t5-base` vs `flan-t5-large` on summarization quality.
3. Prompt the model to output **JSON** and parse it — a common production pattern.